In [1]:
!pip -q install -U \
  "transformers>=4.41,<4.58" accelerate bitsandbytes \
  "huggingface_hub>=0.34,<1.0" \
  sentence-transformers faiss-cpu tqdm rank-bm25 pandas

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 108.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Mounted at /content/drive


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
import os
from pathlib import Path

DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"
print("DATA_DIR =", DATA_DIR)
print("Existe ?", os.path.exists(DATA_DIR))

# Lister rapidement le contenu
print("\nContenu (20 premiers):")
print(os.listdir(DATA_DIR)[:20])

# Compter les fichiers JSON (récursif)
json_files = sorted([str(p) for p in Path(DATA_DIR).rglob("*.json")])
json_files_upper = sorted([str(p) for p in Path(DATA_DIR).rglob("*.JSON")])
print("\nNb .json :", len(json_files))
print("Nb .JSON :", len(json_files_upper))

# Montrer un exemple
example_list = json_files or json_files_upper
print("\nExemple:", example_list[0] if example_list else "Aucun JSON trouvé")

DATA_DIR = /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full
Existe ? True

Contenu (20 premiers):
['page_009.json', 'page_026.json', 'page_048.json', 'page_102.json', 'page_049.json', 'page_072.json', 'page_065.json', 'page_056.json', 'page_080.json', 'page_051.json', 'page_019.json', 'page_046.json', 'page_121.json', 'page_099.json', 'page_052.json', 'page_110.json', 'page_115.json', 'page_103.json', 'page_100.json', 'page_078.json']

Nb .json : 139
Nb .JSON : 0

Exemple: /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/guide_candidat_2025.json


In [4]:
import json

path0 = (json_files or json_files_upper)[0]
with open(path0, "r", encoding="utf-8") as f:
    sample = json.load(f)

print("Fichier:", path0)
print("Type:", type(sample))
if isinstance(sample, dict):
    print("Keys:", list(sample.keys())[:50])
else:
    print("Longueur liste:", len(sample))
    print("Keys du premier item:", list(sample[0].keys())[:50])

Fichier: /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/guide_candidat_2025.json
Type: <class 'dict'>
Keys: ['source', 'url', 'source_file', 'pages', 'sections']


In [5]:
import os, json, re
from pathlib import Path

def norm(x) -> str:
    """
    Normalise n'importe quel type vers une string 'propre'.
    - str -> normalisation whitespace
    - list/dict -> json stringifié
    - autres -> str(...)
    """
    if x is None:
        return ""
    if isinstance(x, str):
        s = x
    elif isinstance(x, (dict, list)):
        s = json.dumps(x, ensure_ascii=False)
    else:
        s = str(x)
    return re.sub(r"\s+", " ", s).strip()

def guess_kind(filename: str) -> str:
    fn = filename.lower()
    if fn.startswith("page_"):
        return "concours"
    if "deroul" in fn or "déroul" in fn or "process" in fn:
        return "general_process"
    if "avantage" in fn or "remuner" in fn or "rémun" in fn or "accompagner" in fn or "carriere" in fn or "carrière" in fn:
        return "general_career"
    if "institut" in fn or "cnrs" in fn:
        return "general_cnrs"
    return "general_other"

In [6]:
def extract_passages(obj: dict, filename: str):
    kind = guess_kind(filename)
    source = obj.get("url") or obj.get("source_url") or obj.get("source_file") or f"local://{filename}"

    passages = []

    # 1) Format "pages": liste de pages/sections
    if isinstance(obj.get("pages"), list):
        for i, p in enumerate(obj["pages"], start=1):
            # p peut être dict ou str
            if isinstance(p, dict):
                sec = p.get("title") or p.get("heading") or p.get("section") or f"Page {i}"
                txt = p.get("text") or p.get("content") or p.get("body") or ""
            else:
                sec = f"Page {i}"
                txt = str(p)
            txt = norm(txt)
            if txt:
                passages.append({
                    "text": txt,
                    "source": source,
                    "section": norm(sec),
                    "doc_id": filename,
                    "title": obj.get("title") or filename,
                    "kind": kind
                })
        return passages

    # 2) Format "institutes": liste (institut CNRS)
    if isinstance(obj.get("institutes"), list):
        for inst in obj["institutes"]:
            if not isinstance(inst, dict):
                continue
            name = inst.get("name") or inst.get("title") or inst.get("acronym") or "Institut"
            desc = inst.get("description") or inst.get("text") or inst.get("content") or ""
            desc = norm(desc)
            if desc:
                passages.append({
                    "text": desc,
                    "source": source,
                    "section": f"Institut: {norm(name)}",
                    "doc_id": filename,
                    "title": obj.get("title") or "Instituts CNRS",
                    "kind": kind
                })
        return passages

    # 3) Format concours / général : champs texte classiques
    title = obj.get("title") or obj.get("intitule") or obj.get("nom") or filename
    # Certains JSON ont des sections structurées
    if isinstance(obj.get("sections"), list):
        for sec in obj["sections"]:
            if not isinstance(sec, dict):
                continue
            sec_title = sec.get("title") or sec.get("heading") or sec.get("section") or "Section"
            sec_text = sec.get("text") or sec.get("content") or sec.get("body") or ""
            sec_text = norm(sec_text)
            if sec_text:
                passages.append({
                    "text": sec_text,
                    "source": source,
                    "section": norm(sec_title),
                    "doc_id": filename,
                    "title": title,
                    "kind": kind
                })
        return passages

    # 4) Fallback texte brut
    text = obj.get("text") or obj.get("content") or obj.get("body") or obj.get("texte") or ""
    text = norm(text)
    if text:
        passages.append({
            "text": text,
            "source": source,
            "section": "Document",
            "doc_id": filename,
            "title": title,
            "kind": kind
        })
        return passages

    # 5) Dernier recours: stringify propre (évite de perdre des infos)
    blob = norm(json.dumps(obj, ensure_ascii=False))
    if blob:
        passages.append({
            "text": blob,
            "source": source,
            "section": "Document (json)",
            "doc_id": filename,
            "title": title,
            "kind": kind
        })
    return passages

In [7]:
json_paths = sorted([str(p) for p in Path(DATA_DIR).rglob("*.json")])

passages = []
kinds_count = {}

for path in json_paths:
    fn = os.path.basename(path)
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        ps = extract_passages(obj, fn)
    elif isinstance(obj, list):
        ps = []
        for item in obj:
            if isinstance(item, dict):
                ps.extend(extract_passages(item, fn))
    else:
        ps = []
    passages.extend(ps)

for p in passages:
    kinds_count[p["kind"]] = kinds_count.get(p["kind"], 0) + 1

print("✅ Passages total:", len(passages))
print("Répartition kinds:", kinds_count)

# exemples
for ex in passages[:3]:
    print("\n---", ex["kind"], "|", ex["doc_id"], "|", ex["section"])
    print("source:", ex["source"])
    print(ex["text"][:250], "...")

✅ Passages total: 170
Répartition kinds: {'general_other': 33, 'general_cnrs': 4, 'concours': 122, 'general_career': 11}

--- general_other | guide_candidat_2025.json | Page 1
source: Guide candidat 2025.pdf
CONCOURS EXTERNES DES PERSONNELS INGÉNIEURS ET TECHNICIENS Le guide du candidat et de la candidate Edition 2025 ...

--- general_other | guide_candidat_2025.json | Page 2
source: Guide candidat 2025.pdf
Direction de la publication : Antoine Petit Direction de la rédaction : Hélène Maury Direction adjointe de la rédaction : Christiane Ename – Laetitia Navarro -Service recrutement et intégration (SeRI) Autrices : Dominique Marx - Emilie Faure - Nathal ...

--- general_other | guide_candidat_2025.json | Page 3
source: Guide candidat 2025.pdf
5 - 6 Pourquoi candidater ? 7 - 8 Le choix des concours 9 - 10 L’inscription 11 Comment concourir ? 12 Les conditions pour concourir 13 - 14 Le déroulement des concours 15-16 Les épreuves 17 La publication des résultats 18 La rémunération 19 RGPD 